[Reference](https://medium.com/@mauryaanoop3/how-to-use-ollama-ocr-with-microsoft-autogen-e533c95f11fc)

In [1]:
pip install -q ollama-ocr autogen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 685.4/685.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.3 MB/s eta 0:00:00


In [2]:
from autogen import AssistantAgent, UserProxyAgent
from autogen import register_function
from ollama_ocr import OCRProcessor

In [3]:
def doc_parser(file_path: str) -> str:
    ocr = OCRProcessor(model_name='granite3.2-vision')
    result = ocr.process_image(
        image_path=file_path,
        format_type="text",
        language="eng",
    )
    return result

In [4]:
config_list = [
    {
        "model": "llama3.2",
        "base_url": "http://localhost:11434/v1",
        'api_key': 'ollama',
    },
]

llm_config = {"config_list": config_list, "cache_seed": 42}

In [5]:
user = UserProxyAgent(
    name="human",
    llm_config=False,
    is_termination_msg=lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"],
    human_input_mode="NEVER",
    code_execution_config=False,
)

In [6]:
ocr_agent = AssistantAgent(
    name="OCR_Agent",
    system_message="You are an expert OCR assistant. "
    "Your primary task is to extract text from documents using the 'doc_parser' tool. "
    "You should call the 'doc_parser' tool with the correct file path. "
    "Once you have extracted the text, summarize the document in no more than 50 words."
    "Return 'TERMINATE' when the task is done.",
    llm_config=llm_config,
    code_execution_config=False,
)

In [7]:
register_function(
    doc_parser,
    caller=ocr_agent,
    executor=user,
    name="doc_parser",
    description="Extract text from a document and return the complete extracted text.",
)

In [9]:
user.initiate_chat(
    ocr_agent,
    message="Hello, I have a document that I need help extracting text from 'panel_ui.pdf' ",
)